Great — this is the **final major upgrade in ray tracing cameras: defocus blur (depth of field)**.

I’ll explain it clearly, then give you a **drop-in Python implementation compatible with your current camera**.

---

# 🎯 IDEA OF DEFOCUS BLUR (SIMPLE INTUITION)

### 📷 Normal camera (your current one)

* Ray starts from a **single point (camera center)**
* Goes through pixel
* Everything sharp everywhere (no blur)

---

### 📷 Real camera

* Ray starts from a **lens (circle area)**
* Not a single point
* Different starting points → different directions

➡ Result:

* Near focus plane = sharp
* Away from focus plane = blurred

---

# 📐 KEY CONCEPTS

## 1. Focus plane

$$
\text{focus plane distance} = f
$$

Everything at distance **f** is sharp.

---

## 2. Aperture (lens radius)

$$
\text{lens radius} = r
$$

* small r → sharp image
* big r → strong blur

---

## 3. Thin lens model idea

Instead of simulating optics:

➡ We sample a random point on a disk

---

# 🧠 GEOMETRY CHANGE (IMPORTANT)

### BEFORE (your camera)

```
ray origin = camera center
```

### NOW (defocus blur)

```
ray origin = random point on lens disk
```

---

# 🎥 WHAT CHANGES IN CAMERA

You ONLY modify 3 things:

---

## 1. Add aperture + focus distance

```python
self.defocus_angle = 0.0  # in degrees (0 = no blur)
self.focus_dist = 1.0
```

---

## 2. Build lens disk basis

We use camera axes:

* u → right
* v → up

Lens lies on u-v plane

---

## 3. Random lens sampling

---

# ✅ FINAL PYTHON CODE (DEFOCUS BLUR ADDED)

Below is the **exact upgrade to your camera class**.

---

## 🔥 ADD THIS TO `__init__`

```python
# ==========================
# DEFOCUS BLUR SETTINGS
# ==========================
self.defocus_angle = 0.0   # 0 = pinhole camera (no blur)
self.focus_dist = 1.0
```

---

## 🔥 ADD THIS TO `initialize()`

### after u, v, w are computed:

```python
# ==========================
# DEFOCUS DISK (LENS)
# ==========================

# lens radius from angle
defocus_radius = self.focus_dist * math.tan(
    math.radians(self.defocus_angle / 2)
)

# lens basis vectors
self.defocus_u = self.u * defocus_radius
self.defocus_v = self.v * defocus_radius
```

---

## 🔥 ADD THIS FUNCTION

```python
def random_in_unit_disk(self):
    while True:
        p = Vec3(
            random.random() * 2 - 1,
            random.random() * 2 - 1,
            0
        )
        if p.x * p.x + p.y * p.y < 1:
            return p
```

---

## 🔥 MODIFY `get_ray()`

### OLD:

```python
origin = self.center
```

---

### NEW (DEFOCUS BLUR ENABLED):

```python
def get_ray(self, i, j):

    offset = self.sample_square()

    pixel_center = (
        self.pixel00_loc
        + (i + offset.x) * self.pixel_delta_u
        + (j + offset.y) * self.pixel_delta_v
    )

    direction = pixel_center - self.center

    # ==========================
    # DEFOCUS BLUR (NEW)
    # ==========================
    if self.defocus_angle <= 0:
        origin = self.center
    else:
        disk = self.random_in_unit_disk()
        origin = (
            self.center
            + self.defocus_u * disk.x
            + self.defocus_v * disk.y
        )

    return Ray(origin, direction)
```

---

# 🧠 WHAT CHANGED (VERY IMPORTANT)

| Feature    | Before                  | After            |
| ---------- | ----------------------- | ---------------- |
| Ray origin | fixed point             | random disk      |
| Sharpness  | infinite depth of field | focus plane only |
| Camera     | pinhole model           | thin lens model  |
| Blur       | none                    | realistic blur   |

---

# 📐 FINAL FORMULA SUMMARY

## Lens sampling:

$$
origin = center + u \cdot x + v \cdot y
$$

where:

* $(x,y)$ random point in unit disk

---

## Ray direction:

$$
direction = pixel_point - origin
$$

---

# 🎯 VISUAL EFFECT

### Without defocus blur:

* everything sharp

### With defocus blur:

* center objects sharp
* background blurred
* foreground blurred

---

# 🚀 NEXT LEVEL (if you want)

You can upgrade this further into:

### 🎬 Real camera system

* autofocus simulation
* moving lens
* bokeh shape (hexagon, iris blades)

### 🎮 cinematic rendering

* depth-of-field animation
* rack focus (focus shifting between objects)

Just tell 👍
